# 01 Dataset Overview

This notebook verifies the raw DREAMT files before downstream modeling. It checks file availability, expected columns, label and event annotation fields, missingness, approximate recording duration, and sleep-stage target standardization. It also creates or reloads the fixed participant-level split used throughout the project.

This notebook intentionally avoids predictive EDA, feature engineering, scaling, and modeling.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_INTERIM_DATA_DIR,
    DEFAULT_RAW_DATA_DIR,
    DEFAULT_SPLIT_ASSIGNMENTS_PATH,
    EVENT_ANNOTATION_COLUMNS,
    EXPECTED_DREAMT_COLUMNS,
    EXPECTED_SIGNAL_COLUMNS,
    LABEL_COLUMN,
    check_no_participant_overlap,
    create_participant_split,
    extract_participant_id,
    list_participant_csvs,
    load_split_assignments,
    save_split_assignments,
    summarize_dataset,
    summarize_split_label_distribution,
)

from src.plots import plot_class_balance, summarize_missingness_by_group

from src.preprocessing import (
    TARGET_SLEEP_STAGE_LABELS,
    identify_invalid_labels,
    map_sleep_stage,
    summarize_label_mapping,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

raw_data_dir = repo_root / DEFAULT_RAW_DATA_DIR
summary_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "participant_summary.csv"
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
split_assignments_path = repo_root / DEFAULT_SPLIT_ASSIGNMENTS_PATH
label_mapping_summary_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "label_mapping_summary.csv"
label_mapping_summary_p_as_wake_path = repo_root / DEFAULT_INTERIM_DATA_DIR / "label_mapping_summary_p_as_wake.csv"
expected_n_participants = 100

preflight_participant_files = (
    list_participant_csvs(raw_data_dir) if raw_data_dir.exists() and raw_data_dir.is_dir() else []
)
display(pd.DataFrame([
    {
        "repo_root": str(repo_root),
        "raw_data_dir": str(raw_data_dir),
        "raw_data_dir_exists": raw_data_dir.exists(),
        "participant_files_found": len(preflight_participant_files),
        "participant_summary_exists": summary_path.exists(),
        "label_mapping_summary_exists": label_mapping_summary_path.exists(),
        "split_assignments_exists": split_assignments_path.exists(),
    }
]))

## Locate Participant Files

This section checks whether the local DREAMT participant CSV files are available under `data/raw/` and identifies files that match the expected participant filename pattern.

The goal is to confirm that the notebook can access the raw data needed to build local summaries. If `data/raw/` is missing, the notebook continues with an empty file list so later sections can either load cached intermediates or display clear fallback messages.

In [ ]:
try:
    participant_files = list_participant_csvs(raw_data_dir)
except FileNotFoundError as exc:
    participant_files = []
    print(exc)

participant_file_table = pd.DataFrame(
    {"file_path": [str(path) for path in participant_files]}
)
print(f"Found {len(participant_files)} participant CSV file(s).")
participant_file_table.head()

## Build Or Load Participant Summary

The participant summary is a cached inventory table used by most checks in this notebook. If `data/interim/participant_summary.csv` already exists, the notebook reloads it. Otherwise, it scans the raw participant CSV files and saves the resulting summary.

If neither the raw files nor a saved summary are available, the notebook uses an empty table. Downstream cells then show clear messages indicating which summary-dependent outputs cannot be produced.

In [ ]:
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print(f"Loaded existing summary: {summary_path}")
else:
    try:
        summary_df = summarize_dataset(raw_data_dir, output_path=summary_path)
    except FileNotFoundError as exc:
        print(exc)
        summary_df = pd.DataFrame()

summary_df.head()

In [ ]:
def parse_json_cell(value, default=None):
    if default is None:
        default = {}
    if pd.isna(value):
        return default
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return default


def require_summary(summary):
    if summary.empty:
        print("No participant summary is available yet. Add raw CSVs under data/raw/ and rerun the notebook.")
        return False
    return True

## Participant Count And File Integrity

This section checks whether all 100 expected participant files were found and whether each summarized file matches the expected DREAMT schema.

The first output provides a compact participant count check. The second output lists per-participant row counts, missing or extra columns, and read errors. These checks help identify incomplete downloads, corrupt files, or schema changes before downstream preprocessing.

In [ ]:
participant_count_table = pd.DataFrame(
    [
        {
            "expected_participant_files": expected_n_participants,
            "found_participant_files": len(participant_files),
            "all_100_present": len(participant_files) == expected_n_participants,
        }
    ]
)
participant_count_table

In [ ]:
if require_summary(summary_df):
    integrity_columns = [
        "participant_id",
        "n_rows",
        "n_columns",
        "has_expected_schema",
        "has_all_expected_columns",
        "missing_expected_columns",
        "extra_columns",
        "error",
    ]
    display(summary_df[integrity_columns].sort_values(["error", "n_rows"], na_position="last"))

    schema_summary = summary_df["has_expected_schema"].value_counts(dropna=False).rename_axis("has_expected_schema").reset_index(name="n_participants")
    display(schema_summary)

## Missing Signals, Labels, And Event Annotations

This section checks whether each participant file contains the expected wearable signals, the sleep-stage label column, and event annotation columns used or documented in later stages.

The first output reports signal presence by participant and summarizes how many participants are missing each signal. The second output performs the same check for `Sleep_Stage` and event annotation columns. This helps separate modeling-critical columns from supplemental annotations that are documented but not used as prediction targets.

In [ ]:
if require_summary(summary_df):
    signal_presence_columns = [f"has_{column}" for column in EXPECTED_SIGNAL_COLUMNS]
    signal_presence = summary_df[["participant_id", *signal_presence_columns]].copy()
    display(signal_presence)

    missing_signal_counts = pd.DataFrame(
        {
            "signal": EXPECTED_SIGNAL_COLUMNS,
            "participants_missing_signal": [
                int((~summary_df[f"has_{column}"].fillna(False).astype(bool)).sum())
                for column in EXPECTED_SIGNAL_COLUMNS
            ],
        }
    )
    display(missing_signal_counts)

In [ ]:
if require_summary(summary_df):
    label_and_event_columns = [
        "participant_id",
        "label_column",
        *[f"has_{column}" for column in EVENT_ANNOTATION_COLUMNS],
    ]
    display(summary_df[label_and_event_columns])

    missing_label_count = int((summary_df["label_column"] != LABEL_COLUMN).sum())
    event_presence_counts = pd.DataFrame(
        {
            "column": EVENT_ANNOTATION_COLUMNS,
            "participants_missing_column": [
                int((~summary_df[f"has_{column}"].fillna(False).astype(bool)).sum())
                for column in EVENT_ANNOTATION_COLUMNS
            ],
        }
    )
    display(pd.DataFrame([{"missing_sleep_stage_participants": missing_label_count}]))
    display(event_presence_counts)

## Empty, Corrupted, Or Unusually Short Files

This section flags participant files that could not be read, have zero rows, or are unusually short relative to the rest of the dataset.

The table identifies files that may need attention, and the histogram provides a quick view of row-count variability across participants. If no participant summary is available, the section prints the shared fallback message instead of producing empty diagnostics that could be misleading.

In [ ]:
if require_summary(summary_df):
    row_threshold = summary_df["n_rows"].quantile(0.05) if summary_df["n_rows"].notna().any() else None
    short_or_problem_files = summary_df[
        summary_df["error"].notna()
        | summary_df["n_rows"].fillna(0).eq(0)
        | (summary_df["n_rows"] < row_threshold if row_threshold is not None else False)
    ][["participant_id", "file_path", "n_rows", "recording_duration_seconds", "error"]]
    display(short_or_problem_files)

    summary_df["n_rows"].dropna().plot(kind="hist", bins=30, title="Rows per participant file")
    plt.xlabel("Rows")
    plt.show()

## Sleep-Stage And Event Annotation Values

This section summarizes the raw values observed in `Sleep_Stage` and in each event annotation column.

The goal is to identify which sleep-stage labels need to be mapped or excluded and to document the annotation value vocabulary present in the local files. If the cached summary contains no values for a category, the notebook states this explicitly.

In [ ]:
if require_summary(summary_df):
    label_rows = []
    for _, row in summary_df.iterrows():
        for label_value, count in parse_json_cell(row.get("label_counts"), {}).items():
            label_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "sleep_stage_value": label_value,
                    "count": count,
                }
            )
    label_values = pd.DataFrame(label_rows)
    if not label_values.empty:
        display(label_values.groupby("sleep_stage_value", as_index=False)["count"].sum())
    else:
        print("No Sleep_Stage values found in the participant summary.")

In [ ]:
if require_summary(summary_df):
    event_rows = []
    for _, row in summary_df.iterrows():
        event_counts = parse_json_cell(row.get("event_annotation_value_counts"), {})
        for column, counts in event_counts.items():
            for value, count in counts.items():
                event_rows.append(
                    {
                        "participant_id": row["participant_id"],
                        "event_column": column,
                        "event_value": value,
                        "count": count,
                    }
                )
    event_values = pd.DataFrame(event_rows)
    if not event_values.empty:
        display(event_values.groupby(["event_column", "event_value"], as_index=False)["count"].sum())
    else:
        print("No event annotation values found in the participant summary.")

## Missingness By Signal And Participant

This section summarizes missingness across wearable signals and participants.

It expands the cached per-participant missingness summaries into a participant-by-signal table and a mean-missingness plot. When the full epoch index is available, it also summarizes mean signal missingness by mapped sleep stage and the participant-level distribution of mean signal missingness.

These checks are descriptive only and are not used to guide predictive modeling.

In [ ]:
if require_summary(summary_df):
    missingness_rows = []
    for _, row in summary_df.iterrows():
        percentages = parse_json_cell(row.get("missing_value_percentages_by_signal"), {})
        for signal, missing_pct in percentages.items():
            missingness_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "signal": signal,
                    "missing_pct": missing_pct,
                }
            )
    missingness = pd.DataFrame(missingness_rows)
    if not missingness.empty:
        display(missingness.pivot(index="participant_id", columns="signal", values="missing_pct"))
        missingness.groupby("signal")["missing_pct"].mean().sort_values().plot(kind="barh", title="Mean missingness by signal")
        plt.xlabel("Missing values (%)")
        plt.show()
    else:
        print("No signal missingness values found in the participant summary.")

if epoch_index_path.exists():
    epoch_index = pd.read_csv(epoch_index_path)
    required_columns = {"participant_id", "mapped_label", "is_valid_epoch"}
    missing_columns = sorted(required_columns - set(epoch_index.columns))
    if missing_columns:
        print(f"Epoch index is missing column(s): {missing_columns}; skipping epoch-level missingness tables.")
    else:
        valid_epochs = epoch_index[epoch_index["is_valid_epoch"].astype(bool)].copy()
        valid_epochs = valid_epochs[valid_epochs["mapped_label"].isin(TARGET_SLEEP_STAGE_LABELS)].copy()
        missingness_by_participant = summarize_missingness_by_group(
            valid_epochs,
            "participant_id",
        )
        missingness_by_stage = summarize_missingness_by_group(
            valid_epochs,
            "mapped_label",
        )
        if missingness_by_stage.empty or missingness_by_participant.empty:
            print("No epoch-level missingness values found in the epoch index.")
        else:
            stage_missingness_table = missingness_by_stage.pivot_table(
                index="signal",
                columns="mapped_label",
                values="mean_missingness",
            ).reindex(columns=list(TARGET_SLEEP_STAGE_LABELS))
            display(stage_missingness_table)

            participant_missingness_table = missingness_by_participant.pivot_table(
                index="participant_id",
                columns="signal",
                values="mean_missingness",
            )
            display(participant_missingness_table.describe())
else:
    print(f"Epoch index not found at {epoch_index_path}; skipping epoch-level missingness tables.")


## Approximate Recording Duration

This section converts each participant’s summarized recording duration to hours and plots the distribution across participants.

The goal is to identify unexpectedly short or long recordings that could affect epoch counts and participant-level split balance. The calculation uses cached summary metadata only and does not re-read the raw signal files.

In [ ]:
if require_summary(summary_df):
    duration = summary_df[["participant_id", "recording_duration_seconds"]].copy()
    duration["recording_duration_hours"] = duration["recording_duration_seconds"] / 3600
    display(duration.sort_values("recording_duration_seconds"))

    duration["recording_duration_hours"].dropna().plot(kind="hist", bins=30, title="Recording duration per participant")
    plt.xlabel("Hours")
    plt.show()

## Three-Class Label Mapping

The primary project target has three classes: `Wake`, `Non-REM`, and `REM`. PSG stages `N1`, `N2`, and `N3` are grouped as `Non-REM`.

In the DREAMT `data_64Hz` files, the label `P` indicates preparation before PSG recording starts. The primary mapping excludes `P`, while a secondary sensitivity mapping treats `P` as `Wake` to match the `data_100Hz` convention.

The next cell loads saved mapping summaries when available, creates them from the raw files when needed, or displays a clear message when neither source is available.


In [ ]:
if label_mapping_summary_path.exists() and label_mapping_summary_p_as_wake_path.exists():
    label_mapping_summary = pd.read_csv(label_mapping_summary_path)
    label_mapping_summary_p_as_wake = pd.read_csv(label_mapping_summary_p_as_wake_path)
    print(f"Loaded existing primary label mapping summary: {label_mapping_summary_path}")
    print(f"Loaded existing P-as-Wake sensitivity summary: {label_mapping_summary_p_as_wake_path}")
elif participant_files:
    label_mapping_summary = summarize_label_mapping(
        participant_files,
        p_as_wake=False,
        output_path=label_mapping_summary_path,
    )
    label_mapping_summary_p_as_wake = summarize_label_mapping(
        participant_files,
        p_as_wake=True,
        output_path=label_mapping_summary_p_as_wake_path,
    )
    print(f"Saved primary label mapping summary to {label_mapping_summary_path}")
    print(f"Saved P-as-Wake sensitivity summary to {label_mapping_summary_p_as_wake_path}")
else:
    label_mapping_summary = pd.DataFrame()
    label_mapping_summary_p_as_wake = pd.DataFrame()
    print(
        "No participant files or saved label mapping summaries are available. "
        "Add raw CSVs under data/raw/ and rerun this section."
    )

label_mapping_summary.head()


### Raw And Mapped Label Counts

This section shows how each raw sleep-stage value is mapped to one of the three target classes or assigned an exclusion reason.

The mapped-count table gives the dataset-level target distribution before applying the participant-level split.

In [ ]:
if not label_mapping_summary.empty:
    primary_dataset_rows = label_mapping_summary[label_mapping_summary["scope"] == "dataset"]
    raw_counts = primary_dataset_rows[["raw_label", "standardized_label", "mapped_label", "invalid_reason", "count"]]
    display(raw_counts.sort_values(["invalid_reason", "mapped_label", "raw_label"], na_position="last"))

    mapped_counts = (
        primary_dataset_rows.dropna(subset=["mapped_label"])
        .groupby("mapped_label", as_index=False)["count"]
        .sum()
        .sort_values("mapped_label")
    )
    display(mapped_counts)

    dataset_class_balance = (
        mapped_counts.set_index("mapped_label")
        .reindex(TARGET_SLEEP_STAGE_LABELS, fill_value=0)
        .rename_axis("mapped_label")
        .reset_index()
        .rename(columns={"count": "n_epochs"})
    )
    total_epochs = dataset_class_balance["n_epochs"].sum()
    dataset_class_balance["percentage"] = (
        dataset_class_balance["n_epochs"] / total_epochs * 100 if total_epochs else 0
    )
    fig, ax = plot_class_balance(dataset_class_balance)
    ax.set_ylabel("Epochs")
    ax.set_title("Dataset Class Balance")
    plt.show()
else:
    print("Raw and mapped label counts are unavailable because label_mapping_summary is empty.")


### Exclusions Under Primary Mapping

This section separates preparation-stage `P` labels from labels excluded because they are missing, unknown, or otherwise invalid.

This distinction is important because `P` is handled in a planned sensitivity mapping, while other invalid labels generally remain unusable.

In [ ]:
if not label_mapping_summary.empty:
    primary_dataset_rows = label_mapping_summary[label_mapping_summary["scope"] == "dataset"]
    exclusions = (
        primary_dataset_rows.dropna(subset=["invalid_reason"])
        .groupby("invalid_reason", as_index=False)["count"]
        .sum()
        .sort_values("invalid_reason")
    )
    display(exclusions)

    excluded_due_to_p = int(exclusions.loc[exclusions["invalid_reason"] == "Preparation", "count"].sum())
    excluded_missing_or_other = int(exclusions.loc[exclusions["invalid_reason"] != "Preparation", "count"].sum())
    display(pd.DataFrame([
        {
            "excluded_due_to_P": excluded_due_to_p,
            "excluded_due_to_missing_or_other_invalid": excluded_missing_or_other,
        }
    ]))
else:
    print("Exclusion counts are unavailable because label_mapping_summary is empty.")


### Preparation-Stage Counts By Participant

This table shows which participants have the most preparation-stage P rows under the primary mapping.

The goal is to assess whether preparation-stage data are concentrated in a few participant files or distributed broadly across the dataset.

In [ ]:
if not label_mapping_summary.empty:
    participant_p_counts = (
        label_mapping_summary[
            (label_mapping_summary["scope"] == "participant")
            & (label_mapping_summary["standardized_label"] == "P")
        ][["participant_id", "count"]]
        .rename(columns={"count": "p_count"})
        .sort_values(["p_count", "participant_id"], ascending=[False, True])
    )
    display(participant_p_counts)
else:
    print("Preparation-stage counts are unavailable because label_mapping_summary is empty.")


### Participants With Little Or No Target-Class Coverage

This subsection identifies participants with no rows in at least one target class or very few mapped target rows overall.

These participants may affect class balance, epoch availability, or the stability of participant-level split summaries.

In [ ]:
if not label_mapping_summary.empty:
    participant_target_counts = (
        label_mapping_summary[
            (label_mapping_summary["scope"] == "participant")
            & label_mapping_summary["mapped_label"].notna()
        ]
        .pivot_table(
            index="participant_id",
            columns="mapped_label",
            values="count",
            aggfunc="sum",
            fill_value=0,
        )
        .rename_axis(columns=None)
        .reset_index()
    )
    for target_label in ["Wake", "Non-REM", "REM"]:
        if target_label not in participant_target_counts.columns:
            participant_target_counts[target_label] = 0
    class_totals = participant_target_counts[["Wake", "Non-REM", "REM"]].sum(axis=1)
    low_coverage_threshold = 10
    low_or_missing_class_counts = participant_target_counts[
        (participant_target_counts[["Wake", "Non-REM", "REM"]] == 0).any(axis=1)
        | (class_totals < low_coverage_threshold)
    ].sort_values("participant_id")
    display(low_or_missing_class_counts)
else:
    print("Participant target-class coverage is unavailable because label_mapping_summary is empty.")


### Primary Versus P-As-Wake Sensitivity Counts

This comparison shows how the target distribution changes when preparation-stage `P` rows are treated as `Wake` instead of excluded.

This is a sensitivity check only; it is not the primary modeling definition.

In [ ]:
if not label_mapping_summary.empty and not label_mapping_summary_p_as_wake.empty:
    comparison_rows = []
    for mode_name, summary in [
        ("primary_P_excluded", label_mapping_summary),
        ("sensitivity_P_as_Wake", label_mapping_summary_p_as_wake),
    ]:
        dataset_rows = summary[summary["scope"] == "dataset"]
        class_counts = (
            dataset_rows.dropna(subset=["mapped_label"])
            .groupby("mapped_label")["count"]
            .sum()
            .to_dict()
        )
        exclusion_counts = (
            dataset_rows.dropna(subset=["invalid_reason"])
            .groupby("invalid_reason")["count"]
            .sum()
            .to_dict()
        )
        comparison_rows.append({
            "mapping": mode_name,
            "Wake": class_counts.get("Wake", 0),
            "Non-REM": class_counts.get("Non-REM", 0),
            "REM": class_counts.get("REM", 0),
            "excluded_P": exclusion_counts.get("Preparation", 0),
            "excluded_missing_or_other_invalid": sum(
                count for reason, count in exclusion_counts.items() if reason != "Preparation"
            ),
        })
    display(pd.DataFrame(comparison_rows))
else:
    print(
        "Primary versus P-as-Wake sensitivity counts are unavailable because "
        "one or both label mapping summaries are empty."
    )


## Participant-Level Split

This section creates or loads the fixed participant-level train/validation/test split.

Downstream EDA, preprocessing, feature extraction, and modeling should reuse `data/interim/split_assignments.csv` rather than resplitting participants. If the split file is missing, this notebook creates the default 70/15/15 split only when all 100 expected participant files are present. Otherwise, it avoids creating a partial split.


In [ ]:
split_df = pd.DataFrame(columns=["participant_id", "split"])

if split_assignments_path.exists():
    split_df = load_split_assignments(split_assignments_path)
    print(f"Loaded existing split assignments: {split_assignments_path}")
elif len(participant_files) == expected_n_participants:
    participant_ids = [extract_participant_id(path) for path in participant_files]
    split_df = create_participant_split(participant_ids, random_state=42)
    save_split_assignments(split_df, split_assignments_path)
    print(f"Saved split assignments: {split_assignments_path}")
elif participant_files:
    print(
        "Default 70/15/15 split was not created because "
        f"{len(participant_files)} participant file(s) were found, "
        f"not {expected_n_participants}."
    )
else:
    print("No participant files or saved split assignments are available yet.")

if not split_df.empty:
    check_no_participant_overlap(split_df)
    display(split_df["split"].value_counts().reindex(["train", "validation", "test"]))
    display(split_df.sort_values(["split", "participant_id"]).reset_index(drop=True))


### Split Label Distribution

This section joins the saved participant-level split with the label-mapping summary to show target-label counts for the train, validation, and test sets.

The goal is to check whether the fixed split has reasonable label coverage before it is used by downstream notebooks. If either the split file or label summary is unavailable, the notebook reports which prerequisite is missing.

In [ ]:
if not split_df.empty and not label_mapping_summary.empty:
    split_label_summary = summarize_split_label_distribution(label_mapping_summary, split_df)
    display(split_label_summary)

    split_plot_labels = {
        "train": "Training Set Class Balance",
        "validation": "Validation Set Class Balance",
        "test": "Test Set Class Balance",
    }
    split_axis_labels = {
        "train": "Training epochs",
        "validation": "Validation epochs",
        "test": "Test epochs",
    }
    for split_name, plot_title in split_plot_labels.items():
        split_row = split_label_summary[split_label_summary["split"] == split_name]
        if split_row.empty:
            continue
        split_class_balance = pd.DataFrame(
            {
                "mapped_label": list(TARGET_SLEEP_STAGE_LABELS),
                "n_epochs": [
                    int(split_row[f"{label.replace('-', '_')}_count"].iloc[0])
                    for label in TARGET_SLEEP_STAGE_LABELS
                ],
            }
        )
        total_epochs = split_class_balance["n_epochs"].sum()
        split_class_balance["percentage"] = (
            split_class_balance["n_epochs"] / total_epochs * 100 if total_epochs else 0
        )
        fig, ax = plot_class_balance(split_class_balance)
        ax.set_ylabel(split_axis_labels[split_name])
        ax.set_title(plot_title)
        plt.show()
elif split_df.empty:
    print("Split assignments are not available yet.")
else:
    print("Label mapping summary is not available yet.")
